In [10]:
import sqlite3
import pandas as pd

# Connect to SQLite database
DB_PATH = r"C:\Users\tsmal\OneDrive\Desktop\College\Winter 2026\IS 455\assignments\Deployment Assignment\shop.db"
conn = sqlite3.connect(DB_PATH)

# Optional: pull raw tables if you want to inspect them directly
customers_df = pd.read_sql_query("SELECT * FROM customers", conn)
products_df = pd.read_sql_query("SELECT * FROM products", conn)
orders_df = pd.read_sql_query("SELECT * FROM orders", conn)
order_items_df = pd.read_sql_query("SELECT * FROM order_items", conn)
shipments_df = pd.read_sql_query("SELECT * FROM shipments", conn)
reviews_df = pd.read_sql_query("SELECT * FROM product_reviews", conn)

# Build one row per order with customer + shipping + item-level aggregates
mlr_query = """
WITH item_agg AS (
    SELECT
        oi.order_id,
        SUM(oi.quantity) AS total_units,
        COUNT(*) AS line_items,
        COUNT(DISTINCT oi.product_id) AS distinct_products,
        AVG(oi.unit_price) AS avg_unit_price,
        SUM(oi.line_total) AS line_total_sum
    FROM order_items oi
    GROUP BY oi.order_id
)
SELECT
    o.order_id,
    o.customer_id,
    o.order_datetime,
    o.billing_zip,
    o.shipping_zip,
    o.shipping_state,
    o.payment_method,
    o.device_type,
    o.ip_country,
    o.promo_used,
    o.order_subtotal,
    o.shipping_fee,
    o.tax_amount,
    o.order_total,
    o.risk_score,
    o.is_fraud,
    c.gender,
    c.birthdate,
    c.city,
    c.state AS customer_state,
    c.customer_segment,
    c.loyalty_tier,
    c.is_active AS customer_is_active,
    s.carrier,
    s.shipping_method,
    s.distance_band,
    s.promised_days,
    s.actual_days,
    s.late_delivery,
    ia.total_units,
    ia.line_items,
    ia.distinct_products,
    ia.avg_unit_price,
    ia.line_total_sum
FROM orders o
LEFT JOIN customers c ON o.customer_id = c.customer_id
LEFT JOIN shipments s ON o.order_id = s.order_id
LEFT JOIN item_agg ia ON o.order_id = ia.order_id
"""

mlr_df = pd.read_sql_query(mlr_query, conn)

# Basic feature cleanup/engineering for regression workflows
mlr_df["order_datetime"] = pd.to_datetime(mlr_df["order_datetime"], errors="coerce")
mlr_df["birthdate"] = pd.to_datetime(mlr_df["birthdate"], errors="coerce")
mlr_df["customer_age"] = ((mlr_df["order_datetime"] - mlr_df["birthdate"]).dt.days / 365.25).round(1)
mlr_df["order_hour"] = mlr_df["order_datetime"].dt.hour
mlr_df["order_dayofweek"] = mlr_df["order_datetime"].dt.dayofweek

# Remove high-cardinality IDs and raw date fields to keep modeling cleaner
mlr_df = mlr_df.drop(columns=["order_id", "customer_id", "birthdate"], errors="ignore")

print("MLR dataframe shape:", mlr_df.shape)
print("\nColumns:")
print(mlr_df.columns.tolist())
print("\nSample:")
display(mlr_df.head())

conn.close()

MLR dataframe shape: (5000, 34)

Columns:
['order_datetime', 'billing_zip', 'shipping_zip', 'shipping_state', 'payment_method', 'device_type', 'ip_country', 'promo_used', 'order_subtotal', 'shipping_fee', 'tax_amount', 'order_total', 'risk_score', 'is_fraud', 'gender', 'city', 'customer_state', 'customer_segment', 'loyalty_tier', 'customer_is_active', 'carrier', 'shipping_method', 'distance_band', 'promised_days', 'actual_days', 'late_delivery', 'total_units', 'line_items', 'distinct_products', 'avg_unit_price', 'line_total_sum', 'customer_age', 'order_hour', 'order_dayofweek']

Sample:


,order_datetime,billing_zip,shipping_zip,shipping_state,payment_method,device_type,ip_country,promo_used,order_subtotal,shipping_fee,...,actual_days,late_delivery,total_units,line_items,distinct_products,avg_unit_price,line_total_sum,customer_age,order_hour,order_dayofweek
0,2025-11-29 00:51:07,28289,28289,CO,card,mobile,US,0,662.95,15.44,...,5,1,9,5,5,69.242,662.95,20.5,0,5
1,2025-09-01 10:25:59,28289,13888,NY,card,desktop,US,1,862.92,14.74,...,3,1,7,5,5,133.300,862.92,20.2,10,0
2,2025-12-15 07:24:41,28289,28289,CO,card,mobile,US,0,796.09,14.04,...,8,1,5,3,3,140.850,796.09,20.5,7,0
3,2025-11-06 18:21:19,28289,28289,CO,bank,mobile,US,1,137.60,6.99,...,6,0,1,1,1,137.600,137.60,20.4,18,3
4,2025-11-30 05:34:15,28289,28289,CO,card,mobile,CA,0,17.07,6.99,...,7,1,1,1,1,17.070,17.07,20.5,5,6


In [11]:
# Fraud target sanity check (1 = fraud, 0 = not fraud)
fraud_rate = mlr_df["is_fraud"].mean()
print(f"Fraud rate: {fraud_rate:.2%}")
print(mlr_df["is_fraud"].value_counts(dropna=False))

Fraud rate: 6.36%
is_fraud
0    4682
1     318
Name: count, dtype: int64


In [12]:
import os
from datetime import datetime

import joblib
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Split features/target
X = mlr_df.drop(columns=["is_fraud"])
y = mlr_df["is_fraud"].astype(int)

# Remove raw datetime field (we already engineered hour/day features)
X = X.drop(columns=["order_datetime"], errors="ignore")

categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_cols = [c for c in X.columns if c not in categorical_cols]

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_cols),
        ("cat", categorical_pipeline, categorical_cols),
    ]
)

# Logistic regression for binary fraud classification
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, y_proba), 4))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=4))

# Save versioned model artifact
artifact_dir = "artifacts"
os.makedirs(artifact_dir, exist_ok=True)
version = datetime.now().strftime("%Y%m%d_%H%M%S")
model_path = os.path.join(artifact_dir, f"fraud_logreg_{version}.sav")
joblib.dump(model, model_path)

print(f"\nSaved model: {model_path}")

C:\Users\tsmal\AppData\Local\Temp\ipykernel_15276\2587788566.py:20: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()


Accuracy: 0.922
ROC-AUC: 0.9684

Confusion Matrix:
[[871  65]
 [ 13  51]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9853    0.9306    0.9571       936
           1     0.4397    0.7969    0.5667        64

    accuracy                         0.9220      1000
   macro avg     0.7125    0.8637    0.7619      1000
weighted avg     0.9504    0.9220    0.9322      1000


Saved model: artifacts\fraud_logreg_20260402_085222.sav


In [13]:
# Simulate application restart: load artifact and run inference on new rows
loaded_model = joblib.load(model_path)

sample_rows = X_test.head(5).copy()
sample_pred = loaded_model.predict(sample_rows)
sample_proba = loaded_model.predict_proba(sample_rows)[:, 1]

inference_preview = sample_rows.copy()
inference_preview["pred_is_fraud"] = sample_pred
inference_preview["pred_fraud_probability"] = sample_proba.round(4)

display(inference_preview.head())

,billing_zip,shipping_zip,shipping_state,payment_method,device_type,ip_country,promo_used,order_subtotal,shipping_fee,tax_amount,...,total_units,line_items,distinct_products,avg_unit_price,line_total_sum,customer_age,order_hour,order_dayofweek,pred_is_fraud,pred_fraud_probability
2936,76784,76784,CO,card,tablet,US,0,183.36,7.34,11.60,...,3,3,3,61.120000,183.36,22.0,12,4,0,0.0153
18,28289,28289,CO,paypal,tablet,US,1,102.65,7.34,5.89,...,3,3,3,34.216667,102.65,20.2,4,2,0,0.0044
580,28289,28289,CO,card,desktop,US,0,495.55,7.69,38.06,...,4,4,4,123.887500,495.55,20.2,6,4,0,0.2373
4405,41946,41946,PA,card,mobile,US,0,317.67,7.34,23.40,...,3,2,2,87.315000,317.67,62.4,16,2,0,0.0000
4561,62184,62184,VA,card,mobile,US,0,940.70,7.69,55.89,...,4,4,4,235.175000,940.70,20.2,19,6,1,0.8474


In [14]:
# Shared utilities for model comparison
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, log_loss

# Reuse X_train, X_test, y_train, y_test and preprocessor from earlier cells
comparison_results = {}

def evaluate_classifier(name, fitted_pipeline):
    y_pred = fitted_pipeline.predict(X_test)
    y_proba = fitted_pipeline.predict_proba(X_test)

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "log_loss": log_loss(y_test, y_proba),
        "f1": f1_score(y_test, y_pred),
    }
    comparison_results[name] = metrics

    print(f"{name} metrics")
    print(f"Accuracy: {metrics['accuracy']:.4f}")
    print(f"Log Loss: {metrics['log_loss']:.4f}")
    print(f"F1 Score: {metrics['f1']:.4f}")
    return metrics

In [15]:
# Logistic Regression (baseline)
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

log_reg_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
    ]
)

log_reg_model.fit(X_train, y_train)
_ = evaluate_classifier("LogisticRegression", log_reg_model)

LogisticRegression metrics
Accuracy: 0.9220
Log Loss: 0.1910
F1 Score: 0.5667


In [16]:
# Random Forest
rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=300,
                max_depth=None,
                min_samples_split=5,
                min_samples_leaf=2,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

rf_model.fit(X_train, y_train)
_ = evaluate_classifier("RandomForest", rf_model)

RandomForest metrics
Accuracy: 0.9230
Log Loss: 0.2507
F1 Score: 0.3304


In [17]:
# XGBoost
# If this import fails, install with: pip install xgboost
from xgboost import XGBClassifier

xgb_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            XGBClassifier(
                n_estimators=300,
                learning_rate=0.05,
                max_depth=6,
                subsample=0.9,
                colsample_bytree=0.9,
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

xgb_model.fit(X_train, y_train)
_ = evaluate_classifier("XGBoost", xgb_model)

XGBoost metrics
Accuracy: 0.9580
Log Loss: 0.1029
F1 Score: 0.6038


In [18]:
# Compare models and identify best
comparison_df = pd.DataFrame(comparison_results).T
comparison_df = comparison_df[["accuracy", "log_loss", "f1"]]

# For fraud classification, prioritize F1 (class balance between precision and recall)
comparison_df = comparison_df.sort_values(by=["f1", "log_loss", "accuracy"], ascending=[False, True, False])

display(comparison_df)

best_model_name = comparison_df.index[0]
print(f"Best model for this task (by F1, with log loss tie-break): {best_model_name}")

,accuracy,log_loss,f1
XGBoost,0.958,0.102893,0.603774
LogisticRegression,0.922,0.190967,0.566667
RandomForest,0.923,0.250722,0.330435


Best model for this task (by F1, with log loss tie-break): XGBoost


In [19]:
# Save artifact
MODEL_PATH = "fraud_xgboost_model.sav"
joblib.dump(xgb_model, MODEL_PATH)
print("Saved:", MODEL_PATH)

Saved: fraud_xgboost_model.sav
